In [8]:
import os
import sys
import difflib
import importlib

PARSER_MODULE_NAME = 'pds_parser'
print("--- Initializing Test Environment ---")
current_working_directory = os.getcwd()
if current_working_directory not in sys.path:
    sys.path.insert(0, current_working_directory)
print(f"Current working directory: {current_working_directory}")

if PARSER_MODULE_NAME in sys.modules:
    try:
        parser_module_reloaded = importlib.reload(sys.modules[PARSER_MODULE_NAME])
        print(f"--- Module '{PARSER_MODULE_NAME}' reloaded. ---")
        from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsComment, PdsBlankLine, PdsList
    except Exception as e:
        print(f"--- FAILED to reload module '{PARSER_MODULE_NAME}': {e} ---")
        from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsComment, PdsBlankLine, PdsList
else:
    from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsComment, PdsBlankLine, PdsList
    print(f"--- Module '{PARSER_MODULE_NAME}' imported fresh. ---")

print(f"--- Using PdsParser from: {PdsParser.__module__}.py ---")
print("-" * 80)

test_content = """
################
# WITCH EVENTS #
################

namespace = witch

#witch.1001-1999 - Guardian coverts ward
#witch.2001-2899 - Convert to witchcraft scheme

witch.1001 = { #by Mathilda Bjarnehed
    hidden = yes
    
    trigger = {
        is_witch_trigger = no
        any_relation = {
            type = guardian
            is_witch_trigger = yes
        }
    }

    immediate = {
        save_scope_as = child
        if = { # Conditional block
            limit = {
                is_ai = yes
                exists = house
                house = {
                    has_house_modifier = witch_coven
                    house_head = { is_ai = yes }
                }
                any_relation = {
                    type = guardian
                    is_ai = yes
                }
            }
            child_witch_conversion_success_effect = yes
        }
        else = {
            random_relation = { type = guardian trigger_event = witch.1002 }
        }
    }
}   

scripted_trigger witch_1002_allow_reveal_outcome_trigger = {
    exists = scope:child.liege
    scope:guardian = {
        NOT = { this = scope:child.liege }
        any_secret = {
            secret_type = secret_witch
            OR = {
                NOT = { is_known_by = scope:child }
                NOT = { is_known_by = scope:child.liege }
            }
        }
    }
}


# Standard Values
@pos_compat_high = 30
@pos_compat_medium = 15
@pos_compat_low = 5

# INTRIGUE OUTCOMES
education_intrigue_1 = {
    minimum_age = 16
    intrigue = 2
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.1
    
    ruler_designer_cost = 0
    
    culture_modifier = {
        parameter = poorly_educated_leaders_distrusted
        feudal_government_opinion = -10
    }
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_1_desc
            }
            desc = trait_education_intrigue_1_character_desc
        }
    }

    group = education_intrigue
    level = 1
}
education_intrigue_2 = {
    minimum_age = 16
    intrigue = 4
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.2
    
    ruler_designer_cost = 20
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_2_desc
            }
            desc = trait_education_intrigue_2_character_desc
        }
    }

    group = education_intrigue
    level = 2
}
"""

test_filepath = "test_events_parser_final.txt"
with open(test_filepath, "w", encoding="utf-8-sig") as f:
    f.write(test_content)
print(f"--- Created test file: {test_filepath} ---")
print("-" * 80)

parser = PdsParser()
print(f"--- Parsing '{test_filepath}'... ---")
parsed_nodes = parser.parse_file(test_filepath)

if parsed_nodes:
    print("\n--- Parsed Tree (Root Nodes Summary) ---")
    for i, node in enumerate(parsed_nodes):
        print(f"{i:03d}: {node!r}")
        if isinstance(node, PdsBlock) and node.children:
            # Limited child printing for brevity
            # pass 
            # To see more children:
            # print(f"     Children of '{node.key}' (first 5):")
            # for j, child_node in enumerate(node.children[:5]):
            #      print(f"       {j:03d}: {child_node!r}")
            # if len(node.children) > 5:
            #     print(f"       ... and {len(node.children) - 5} more children.")
            pass # Keep summary concise for now
    print("-" * 80)

    reconstructed_content = parser.to_string()
    
    # --- Detailed Diffing and Verification ---
    # Using splitlines(True) for difflib to handle newlines consistently if possible
    original_lines_for_diff = test_content.splitlines(True)
    reconstructed_lines_for_diff = reconstructed_content.splitlines(True)

    # Remove initial blank line from test_content if it exists and reconstruct doesn't make one
    if original_lines_for_diff and original_lines_for_diff[0].strip() == "":
        original_lines_for_diff = original_lines_for_diff[1:]
    if reconstructed_lines_for_diff and reconstructed_lines_for_diff[0].strip() == "":
         reconstructed_lines_for_diff = reconstructed_lines_for_diff[1:]

    diff = list(difflib.unified_diff(original_lines_for_diff, reconstructed_lines_for_diff,
                                     fromfile='Original', tofile='Reconstructed', lineterm='', n=3))
    
    # Check if the diff list contains any actual difference lines (starting with '+' or '-')
    # excluding the header lines '--- Original' and '+++ Reconstructed' and '@@ ... @@'
    actual_diff_lines = [d_line for d_line in diff if (d_line.startswith('+') or d_line.startswith('-')) and \
                                                    not d_line.startswith('---') and not d_line.startswith('+++')]

    if not actual_diff_lines:
        print("\nSUCCESS: Reconstructed content PERFECTLY matches original (or only whitespace/newline normalizations not caught by this diff).")
    else:
        print("\nWARNING: Reconstructed content differences found. Diff printed below:")
        print("Legend: '-' Original, '+' Reconstructed. Context lines are unchanged.")
        print("\n" + "="*70 + " DIFF OUTPUT " + "="*70)
        for line_diff in diff: # Print all diff lines including headers and context
            sys.stdout.write(line_diff) # Use sys.stdout.write to preserve exact line endings from diff
        print("="*70 + " END DIFF " + "="*70)

        recon_file = "reconstructed_parser_final_output.txt"
        orig_file = "original_parser_final_input.txt"
        with open(recon_file, "w", encoding="utf-8-sig") as f: f.write(reconstructed_content)
        with open(orig_file, "w", encoding="utf-8-sig") as f: f.write(test_content)
        print(f"\nFor detailed comparison, see '{orig_file}' and '{recon_file}'")
else:
    print(f"ERROR: No nodes parsed from {test_filepath}.")

# --- Test find_node (example) ---
if parsed_nodes:
    print("\n--- Testing find_node ---")
    test_block_for_find = next((n for n in parsed_nodes if isinstance(n, PdsBlock) and n.key == 'witch.1001'), None)
    if test_block_for_find:
        print(f"Searching in block: '{test_block_for_find.key}' (L{test_block_for_find.line_number})")
        path1 = 'trigger.is_witch_trigger'
        found_node1 = test_block_for_find.find_node(path1)
        print(f"Finding '{path1}': {found_node1!r}" + (f" | Value: '{found_node1.value}'" if hasattr(found_node1, 'value') else ""))
        path2 = 'trigger.any_relation.type'
        found_node2 = test_block_for_find.find_node(path2)
        print(f"Finding '{path2}': {found_node2!r}" + (f" | Value: '{found_node2.value}'" if hasattr(found_node2, 'value') else ""))
        path3 = 'immediate.if.limit.house.house_head' # Corrected path
        found_node3 = test_block_for_find.find_node(path3)
        print(f"Finding '{path3}': {found_node3!r}" + (f" | Value: '{found_node3.value}'" if hasattr(found_node3, 'value') else ""))

    target_key = 'scripted_trigger witch_1002_allow_reveal_outcome_trigger'
    scripted_trigger_node = next((n for n in parsed_nodes if hasattr(n, 'key') and n.key == target_key), None)
    if scripted_trigger_node:
        print(f"Found root node '{target_key}': {scripted_trigger_node!r}")
        if isinstance(scripted_trigger_node, PdsBlock):
            exists_node = scripted_trigger_node.find_node('exists')
            print(f"  Finding 'exists' in it: {exists_node!r}" + (f" | Value: '{exists_node.value}'" if hasattr(exists_node, 'value') else ""))
print("\n" + "="*80)
print("Parser Test Run Complete.")
print("="*80)

--- Initializing Test Environment ---
Current working directory: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update
--- Module 'pds_parser' reloaded. ---
--- Using PdsParser from: pds_parser.py ---
--------------------------------------------------------------------------------
--- Created test file: test_events_parser_final.txt ---
--------------------------------------------------------------------------------
--- Parsing 'test_events_parser_final.txt'... ---

--- Parsed Tree (Root Nodes Summary) ---
000: <PdsBlankLine L1 C1 Key='Blank Line'>
001: <PdsComment L2 C1 Key='Comment: '###############...''>
002: <PdsComment L3 C1 Key='Comment: 'WITCH EVENTS #...''>
003: <PdsComment L4 C1 Key='Comment: '###############...''>
004: <PdsBlankLine L5 C1 Key='Blank Line'>
005: <PdsKeyValuePair L6 C1 Key='namespace'>
006: <PdsBlankLine L7 C1 Key='Blank Line'>
007: <PdsComment L8 C1 Key='Comment: 'witch.1001-1999 - Guardian cov...''>
008: <PdsComment L9 C1 Key='Comment: 'witch.2001-2899 - Convert to w...

In [ ]:
## LEXER TEST  ##

import os
import sys
import importlib # Added for attempting to reload the module

# --- Configuration ---
LEXER_MODULE_NAME = 'pds_lexer' # Ensure your lexer file is pds_lexer.py

# --- Attempt to reload the lexer module ---
# This is to help ensure the latest version is used, but restarting the kernel is more reliable.
if LEXER_MODULE_NAME in sys.modules:
    try:
        lexer_module = importlib.reload(sys.modules[LEXER_MODULE_NAME])
        print(f"--- Module '{LEXER_MODULE_NAME}' reloaded successfully. ---")
    except Exception as e:
        print(f"--- FAILED to reload module '{LEXER_MODULE_NAME}': {e} ---")
else:
    print(f"--- Module '{LEXER_MODULE_NAME}' not yet imported, will import fresh. ---")

# --- Path setup ---
# Ensure pds_lexer.py is in the Python path
current_working_directory = os.getcwd()
if current_working_directory not in sys.path:
    sys.path.insert(0, current_working_directory)
print(f"--- Current working directory: {current_working_directory} ---")
print(f"--- Python sys.path (first few entries): {sys.path[:3]} ---")

# --- Import Lexer (after potential reload) ---
try:
    from pds_lexer import PdsLexer, PdsToken
    print("--- PdsLexer and PdsToken imported successfully. ---")
except ImportError as e:
    print(f"--- FATAL: Could not import PdsLexer or PdsToken: {e} ---")
    print("--- Please ensure 'pds_lexer.py' is in the same directory or Python path and contains these classes. ---")
    # Stop further execution if import fails
    raise

# --- Test Content ---
test_content = """
################
# WITCH EVENTS #
################

namespace = witch

#witch.1001-1999 - Guardian coverts ward
#witch.2001-2899 - Convert to witchcraft scheme

witch.1001 = { #by Mathilda Bjarnehed
    hidden = yes
    
    trigger = {
        is_witch_trigger = no
        any_relation = {
            type = guardian
            is_witch_trigger = yes
        }
    }

    immediate = {
        save_scope_as = child
        if = { # Conditional block
            limit = {
                is_ai = yes
                exists = house
                house = {
                    has_house_modifier = witch_coven
                    house_head = { is_ai = yes }
                }
                any_relation = {
                    type = guardian
                    is_ai = yes
                }
            }
            child_witch_conversion_success_effect = yes
        }
        else = {
            random_relation = { type = guardian trigger_event = witch.1002 }
        }
    }
}   

scripted_trigger witch_1002_allow_reveal_outcome_trigger = {
    exists = scope:child.liege
    scope:guardian = {
        NOT = { this = scope:child.liege }
        any_secret = {
            secret_type = secret_witch
            OR = {
                NOT = { is_known_by = scope:child }
                NOT = { is_known_by = scope:child.liege }
            }
        }
    }
}


# Standard Values
@pos_compat_high = 30
@pos_compat_medium = 15
@pos_compat_low = 5

# INTRIGUE OUTCOMES
education_intrigue_1 = {
    minimum_age = 16
    intrigue = 2
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.1
    
    ruler_designer_cost = 0
    
    culture_modifier = {
        parameter = poorly_educated_leaders_distrusted
        feudal_government_opinion = -10
    }
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_1_desc
            }
            desc = trait_education_intrigue_1_character_desc
        }
    }

    group = education_intrigue
    level = 1
}
education_intrigue_2 = {
    minimum_age = 16
    intrigue = 4
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.2
    
    ruler_designer_cost = 20
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_2_desc
            }
            desc = trait_education_intrigue_2_character_desc
        }
    }

    group = education_intrigue
    level = 2
}
"""

# --- Helper function to inspect characters ---
def inspect_text_at_location(text, target_line, target_col, window=20):
    """Inspects characters around a given line and column (1-indexed)."""
    print(f"\n--- Inspecting text_content around L{target_line}, C{target_col} ---")
    lines = text.splitlines(True) # Keep line endings
    if not (0 < target_line <= len(lines)):
        print(f"ERROR: Target line {target_line} is out of bounds (1-{len(lines)}).")
        return

    line_content = lines[target_line - 1]
    
    # Adjust column to be 0-indexed for string slicing
    char_index = target_col - 1

    if not (0 <= char_index < len(line_content)):
        print(f"ERROR: Target column {target_col} is out of bounds for line {target_line} (length {len(line_content)}).")
        print(f"Line content: '{line_content.rstrip()}'")
        return

    actual_char = line_content[char_index]
    print(f"Character at L{target_line}, C{target_col}: '{actual_char}' (ord: {ord(actual_char)}, hex: {hex(ord(actual_char))})")

    start = max(0, char_index - window // 2)
    end = min(len(line_content), char_index + window // 2 + 1)
    
    context_before = line_content[start:char_index]
    context_after = line_content[char_index+1:end]
    
    print(f"Context: '{context_before}<HERE>{actual_char}<HERE>{context_after.rstrip()}'")
    print(f"Line (raw): {repr(line_content)}")


# --- Lexer Test Execution ---
print("\n--- Starting Lexer Test ---")

# Inspect the suspected character location IN THE ORIGINAL test_content string
# This helps verify if the input string itself has an anomaly.
# The error is at line 11, col 16.
# Note: Line counting for `test_content` string literal starts after the initial triple quotes.
# The first actual content line "################" is line 2 if we count the initial blank line.
# Let's find the absolute character position for L11, C16 of the *parsed content*.
# The lexer's line counting starts at 1.
# The problematic line in the content block: "witch.1001 = { #by Mathilda Bjarnehed"

# To find the absolute position for `inspect_text_at_location`, we need to be careful.
# Let's assume the lexer's line 11 refers to the 11th non-empty or significant line.
# The content starts with a newline.
# 1: (empty)
# 2: ################
# 3: # WITCH EVENTS #
# 4: ################
# 5: (empty)
# 6: namespace = witch
# 7: (empty)
# 8: #witch.1001-1999 - Guardian coverts ward
# 9: #witch.2001-2899 - Convert to witchcraft scheme
#10: (empty)
#11: witch.1001 = { #by Mathilda Bjarnehed  <-- This is the line
inspect_text_at_location(test_content, 11, 16)


lexer = PdsLexer(test_content)
tokens = [] # Initialize tokens list

try:
    tokens = lexer.tokenize()
    print(f"--- Lexing Complete. Found {len(tokens)} tokens. ---")
    
    if tokens: # Check if tokens list is not empty
        print("\n--- First 20 Tokens: ---")
        for i, token in enumerate(tokens[:20]):
            print(f"{i:03d}: {token}")
        
        print("\n--- Last 20 Tokens (including EOF): ---")
        for i, token in enumerate(tokens[-20:], start=max(0, len(tokens)-20)):
            print(f"{i:03d}: {token}")

        # Example: Find tokens around a specific line where the error might be
        error_line = 11
        print(f"\n--- Tokens around Line {error_line} (and +/- 1 line): ---")
        for token in tokens:
            if error_line -1 <= token.line <= error_line + 1:
                print(token)
    else:
        print("--- No tokens were generated. ---")

except ValueError as e:
    print(f"LEXER ERROR: {e}")
    print("\n--- Lexer state at point of error (if accessible, depends on lexer structure): ---")
    print(f"Lexer Position: {getattr(lexer, 'pos', 'N/A')}")
    print(f"Lexer Line: {getattr(lexer, 'line', 'N/A')}")
    print(f"Lexer Column: {getattr(lexer, 'column', 'N/A')}")
    if hasattr(lexer, 'pos') and hasattr(lexer, 'text'):
        error_pos = lexer.pos
        text_context_start = max(0, error_pos - 30)
        text_context_end = min(len(lexer.text), error_pos + 30)
        print(f"Text context around error (pos {error_pos}):")
        print(f"...'{lexer.text[text_context_start:error_pos]}<ERROR_HERE>{lexer.text[error_pos:text_context_end]}'...")
except Exception as e_general:
    print(f"AN UNEXPECTED ERROR OCCURRED: {e_general}")
    import traceback
    traceback.print_exc()

print("\n--- Lexer Test Complete ---")

--- Module 'pds_lexer' not yet imported, will import fresh. ---
--- Current working directory: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update ---
--- Python sys.path (first few entries): ['c:\\Users\\Galaxy\\LEVI\\jupyter\\ck3_mod_update', 'c:\\Users\\Galaxy\\miniconda3\\python312.zip', 'c:\\Users\\Galaxy\\miniconda3\\DLLs'] ---
--- PdsLexer and PdsToken imported successfully. ---

--- Starting Lexer Test ---

--- Inspecting text_content around L11, C16 ---
Character at L11, C16: '#' (ord: 35, hex: 0x23)
Context: '.1001 = { <HERE>#<HERE>by Mathild'
Line (raw): 'witch.1001 = { #by Mathilda Bjarnehed\n'
--- DEBUGGING BLOCK IN PdsLexer ACTIVATED ---
LEXER_DEBUG: Current char: '#' (Unicode ord: 35), Line: 11, Column: 16, Pos: 177
LEXER_DEBUG: Text context: 'witch.1001 = { <HERE>#by Mathilda Bja'
LEXER_DEBUG: Slice for COMMENT match (first 30 chars): '#by Mathilda Bjarnehed
    hid'
LEXER_DEBUG: Direct test of COMMENT pattern SUCCEEDED. Group(0): '#by Mathilda Bjarnehed'
--- END LEXER DEBUGGIN

In [1]:
# Jupyter Notebook Cell

import os
import sys

# Ensure pds_parser.py and pds_differ.py are in the same directory
# Restart your Jupyter Notebook kernel before running this cell if files have changed!
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsList, PdsComment, PdsBlankLine
from pds_differ import PdsDiffer, PdsChange

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print("-" * 80)

# --- Define Test File Contents ---

# Test Case 1: Simple changes, additions, modifications
# Original state:
old_vanilla_content_1 = """
key_a = value_a
block_b = {
    param_b1 = 10
    param_b2 = text_b2
}
key_c = value_c_old
"""

# Mod's changes: added key_x, modified key_c, modified block_b (param_b1 changed)
mod_content_1 = """
key_a = value_a
block_b = {
    param_b1 = 20 # Mod changed this
    param_b2 = text_b2
    param_b3 = new_param_by_mod # Mod added this
}
key_c = value_c_mod # Mod changed this
key_x = value_x_by_mod # Mod added this
"""

# New Vanilla's changes: modified key_c, added key_y, modified block_b (param_b2 changed)
new_vanilla_content_1 = """
key_a = value_a
block_b = {
    param_b1 = 10
    param_b2 = text_b2_new # NV changed this
    param_b4 = new_param_by_nv # NV added this
}
key_c = value_c_nv # NV changed this (CONFLICT with mod)
key_y = value_y_by_nv # NV added this
"""

# Test Case 2: Deletions, Block type change (simplified)
# old_vanilla: has a block, mod deletes it, new vanilla changes it to a simple KV
old_vanilla_content_2 = """
file_version = 1.0
trait_block = {
    attr_a = 1
    attr_b = 2
}
event_id = 123
"""

# Mod's changes: trait_block is deleted, event_id modified
mod_content_2 = """
file_version = 1.0
event_id = 456_mod # Mod changed this
"""

# New Vanilla's changes: trait_block becomes a KV, event_id changed by NV
new_vanilla_content_2 = """
file_version = 1.1 # NV updated
trait_block = "simplified" # NV changed block to KV
event_id = 789_nv # NV changed this (CONFLICT with mod)
"""


# --- Helper Function to Run Diff ---
def run_and_print_diff(test_name, old_content, mod_content, new_content):
    print(f"\n{'='*20} RUNNING DIFF TEST: {test_name} {'='*20}")
    
    # Write to temporary files for parsing
    with open("temp_old.txt", "w", encoding="utf-8-sig") as f: f.write(old_content)
    with open("temp_mod.txt", "w", encoding="utf-8-sig") as f: f.write(mod_content)
    with open("temp_new.txt", "w", encoding="utf-8-sig") as f: f.write(new_content)

    # Parse files
    parser = PdsParser()
    old_nodes = parser.parse_file("temp_old.txt")
    mod_nodes = parser.parse_file("temp_mod.txt")
    new_nodes = parser.parse_file("temp_new.txt")

    if not old_nodes or not mod_nodes or not new_nodes:
        print(f"ERROR: Failed to parse one or more files for test '{test_name}'.")
        return

    # Run the differ
    differ = PdsDiffer()
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes)

    print(f"\n--- Detected Changes for '{test_name}' ({len(changes)} changes) ---")
    for change in changes:
        print(change)
    print(f"{'='*20} END DIFF TEST: {test_name} {'='*20}\n")

# --- Run Tests ---
run_and_print_diff("Test Case 1: Simple Changes", old_vanilla_content_1, mod_content_1, new_vanilla_content_1)
run_and_print_diff("Test Case 2: Deletions and Type Changes", old_vanilla_content_2, mod_content_2, new_vanilla_content_2)

# Clean up temp files (optional, but good practice)
try:
    os.remove("temp_old.txt")
    os.remove("temp_mod.txt")
    os.remove("temp_new.txt")
except OSError:
    pass # File might not exist if parsing failed

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--------------------------------------------------------------------------------

==================== RUNNING DIFF TEST: Test Case 1: Simple Changes ====================

--- Detected Changes for 'Test Case 1: Simple Changes' (7 changes) ---
PdsChange(Type='MOD_MODIFIED                       ', Path='block_b.param_b1', Parent='block_b', 
          Nodes=[O:param_b1='10', M:param_b1='20', N:param_b1='10'])
PdsChange(Type='VANILLA_MODIFIED                   ', Path='block_b.param_b2', Parent='block_b', 
          Nodes=[O:param_b2='text_b2', M:param_b2='text_b2', N:param_b2='text_b2_new'])
PdsChange(Type='MOD_ADDED                          ', Path='block_b.param_b3', Parent='block_b', 
          Nodes=[O:ABSENT, M:param_b3='new_param_by_mod', N:ABSENT])
PdsChange(Type='VANILLA_ADDED                      ', Path='block_b.param_b4', Parent='block_b', 
          Nodes=[O:

In [1]:
import os
import sys
import difflib
from datetime import datetime

# Add the current directory to Python path to ensure local imports
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# IMPORTANT: Ensure your pds_parser.py and pds_differ.py are the LATEST versions.
# You MUST restart your Jupyter Notebook kernel before running this cell if you've changed those files!
from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsList, PdsComment, PdsOperatorCondition 
from pds_differ import PdsDiffer, PdsChange

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print("-" * 80)

# --- Define Paths to Your Real CK3 Files ---
# IMPORTANT: Replace these with the actual paths on your system
# Ensure these files exist and represent your mod, old vanilla, and new vanilla states.

SIEGE_EVENTS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\events\siege_events.txt"
SIEGE_EVENTS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\events\siege_events.txt"
SIEGE_EVENTS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\events\siege_events.txt"

INNOVATIONS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\common\culture\innovations\00_tribal_innovations.txt"


# Helper function to read file content
def get_file_content(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f:
            return f.read()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception as e_inner:
            print(f"ERROR reading {filepath} with utf-8 fallback: {e_inner}", file=sys.stderr)
            return None
    except FileNotFoundError:
        print(f"WARNING: File not found at {filepath}", file=sys.stderr)
        return None
    except Exception as e:
        print(f"ERROR reading {filepath}: {e}", file=sys.stderr)
        return None

# Helper to write content to a file
def write_to_file(filepath, content):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    try:
        with open(filepath, 'w', encoding='utf-8-sig') as f:
            f.write(content)
        return True
    except Exception as e:
        print(f"ERROR writing to {filepath}: {e}", file=sys.stderr)
        return False

# --- Helper Functions for Tree Manipulation (Copied from mod_updater.py) ---
# These are needed for the simulated merge in the test script.
def find_node_by_path(root_nodes_list, key_path, differ_util): 
    if not key_path: return None
    current_nodes_to_search = root_nodes_list 

    for i, segment in enumerate(key_path):
        found_at_current_level = False
        for node in current_nodes_to_search: 
            node_identifier = differ_util._get_node_identifier(node)
            if node_identifier == segment:
                if i == len(key_path) - 1: 
                    return node 
                elif isinstance(node, PdsBlock): 
                    current_nodes_to_search = node.children 
                    found_at_current_level = True
                    break 
                else: return None 
        if not found_at_current_level: return None 
    return None 

def get_parent_node_by_path(root_nodes_list, key_path, differ_util): 
    if not key_path or len(key_path) < 1: return None, None
    if len(key_path) == 1: # Top-level node, its "parent" is the root_nodes_list itself
        return root_nodes_list, key_path[0] 

    parent_path = key_path[:-1] 
    child_identifier = key_path[-1]

    parent_block = find_node_by_path(root_nodes_list, parent_path, differ_util)
    
    if isinstance(parent_block, PdsBlock): 
        return parent_block, child_identifier
    return None, None 

# Helper Function to Run Diff (updated to output files)
def run_and_print_diff(test_name, old_filepath, mod_filepath, new_filepath):
    print(f"\n{'='*20} RUNNING DIFF TEST: {test_name} {'='*20}")
    
    # Define output directory for this test run
    test_output_dir = os.path.join(os.getcwd(), "test_output", test_name.replace(" ", "_").replace("(", "").replace(")", ""), datetime.now().strftime("%Y%m%d_%H%M%S"))
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir}")

    old_content_raw = get_file_content(old_filepath)
    mod_content_raw = get_file_content(mod_filepath)
    new_content_raw = get_file_content(new_filepath)

    # Save raw files to output for reference
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RAW.txt")), old_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RAW.txt")), mod_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RAW.txt")), new_content_raw or "")
    print(f"  Raw files saved: _OLD_RAW.txt, _MOD_RAW.txt, _NEW_RAW.txt")

    # Parse files
    parser = PdsParser()
    old_nodes = parser.parse_file(old_filepath)
    mod_nodes = parser.parse_file(mod_filepath)
    new_nodes = parser.parse_file(new_filepath)

    # Ensure empty lists if files were not found/parsed for diffing
    if old_nodes is None: old_nodes = []
    if mod_nodes is None: mod_nodes = []
    if new_nodes is None: new_nodes = []

    if not old_nodes and not mod_nodes and not new_nodes:
        print(f"SKIPPING: No content to parse for test '{test_name}' from any source (O, M, N). Check file paths.")
        return

    # Reconstruct parsed content to verify parser
    reconstructed_old = PdsParser._nodes_to_string(old_nodes)
    reconstructed_mod = PdsParser._nodes_to_string(mod_nodes)
    reconstructed_new = PdsParser._nodes_to_string(new_nodes)

    # Save reconstructed files
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RECONSTRUCTED.txt")), reconstructed_old)
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RECONSTRUCTED.txt")), reconstructed_mod)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RECONSTRUCTED.txt")), reconstructed_new)
    print(f"  Reconstructed files saved: _OLD_RECONSTRUCTED.txt, _MOD_RECONSTRUCTED.txt, _NEW_RECONSTRUCTED.txt")

    # Check for perfect reconstruction (crucial for parser robustness)
    old_raw_strip = old_content_raw.strip() if old_content_raw else ""
    mod_raw_strip = mod_content_raw.strip() if mod_content_raw else ""
    new_raw_strip = new_content_raw.strip() if new_content_raw else ""

    if reconstructed_old.strip() != old_raw_strip:
        print(f"\nWARNING: Reconstruction mismatch for OLD file: {os.path.basename(old_filepath)}. Compare _OLD_RAW.txt and _OLD_RECONSTRUCTED.txt")
    if reconstructed_mod.strip() != mod_raw_strip:
        print(f"\nWARNING: Reconstruction mismatch for MOD file: {os.path.basename(mod_filepath)}. Compare _MOD_RAW.txt and _MOD_RECONSTRUCTED.txt")
    if reconstructed_new.strip() != new_raw_strip:
        print(f"\nWARNING: Reconstruction mismatch for NEW file: {os.path.basename(new_filepath)}. Compare _NEW_RAW.txt and _NEW_RECONSTRUCTED.txt")

    # Run the differ
    differ = PdsDiffer()
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes)

    print(f"\n--- DETECTED CHANGES for '{test_name}' ({len(changes)} changes) ---")
    if not changes:
        print("    No significant changes detected (or only identical comments/blank lines were filtered).")
    for change in changes:
        print(change) # PdsChange.__repr__ provides detailed formatting
    
    # --- SIMULATE MERGE (for display purposes only in this test script) ---
    simulated_merged_nodes = [node.copy() for node in new_nodes] # Deep copy new_nodes for modification
    
    # Pass simulated_merged_nodes as the target list
    def _simulate_apply_change(target_nodes_list_root, change_obj, differ_instance_for_helpers):
        parent_nodes_list_or_block, child_id_in_parent = get_parent_node_by_path(target_nodes_list_root, change_obj.key_path, differ_instance_for_helpers)
        
        if parent_nodes_list_or_block is None:
            print(f"  SIMULATED MERGE WARNING: Parent for '{'.'.join(change_obj.key_path)}' not found in target tree. Cannot apply change.")
            return

        sim_comment = f"SimulatedMerge:{datetime.now().strftime('%Y%m%d%H%M%S')}"

        if change_obj.type == 'MOD_ADDED':
            new_node = change_obj.mod_node.copy()
            if hasattr(new_node, 'comment_text_on_line') and new_node.comment_text_on_line is not None: new_node.comment_text_on_line = (new_node.comment_text_on_line + f" {sim_comment} MOD_ADDED")
            else: new_node.comment_text_on_line = f"{sim_comment} MOD_ADDED"
            
            if isinstance(parent_nodes_list_or_block, list): # Root level addition
                parent_nodes_list_or_block.append(new_node)
            else: # Nested addition
                parent_nodes_list_or_block.add_child_at_appropriate_location(new_node)
        elif change_obj.type == 'MOD_MODIFIED':
            new_node = change_obj.mod_node.copy()
            if hasattr(new_node, 'comment_text_on_line') and new_node.comment_text_on_line is not None: new_node.comment_text_on_line = (new_node.comment_text_on_line + f" {sim_comment} MOD_MODIFIED")
            else: new_node.comment_text_on_line = f"{sim_comment} MOD_MODIFIED"

            if isinstance(parent_nodes_list_or_block, list): # Root level modification
                for idx, node in enumerate(parent_nodes_list_or_block):
                    if differ_instance_for_helpers._get_node_identifier(node) == child_id_in_parent:
                        parent_nodes_list_or_block[idx] = new_node
                        break
            else: # Nested modification
                parent_nodes_list_or_block.replace_child(child_id_in_parent, new_node)
        elif change_obj.type == 'VANILLA_ADDED' or change_obj.type == 'VANILLA_MODIFIED':
            node_in_output = find_node_by_path(target_nodes_list_root, change_obj.key_path, differ_instance_for_helpers)
            if node_in_output:
                if hasattr(node_in_output, 'comment_text_on_line') and node_in_output.comment_text_on_line is not None:
                    node_in_output.comment_text_on_line = (node_in_output.comment_text_on_line + f" {sim_comment} NV_MODIFIED")
                else: node_in_output.comment_text_on_line = f"{sim_comment} NV_MODIFIED"
        elif change_obj.type == 'CONFLICT_MODIFIED' or change_obj.type == 'CONFLICT_ADDITION':
            chosen_node = change_obj.mod_node.copy() # Auto-choose MOD's version for simulation
            if hasattr(chosen_node, 'comment_text_on_line') and chosen_node.comment_text_on_line is not None: chosen_node.comment_text_on_line = (chosen_node.comment_text_on_line + f" {sim_comment} CONFLICT_MOD_CHOSEN")
            else: chosen_node.comment_text_on_line = f"{sim_comment} CONFLICT_MOD_CHOSEN"

            if isinstance(parent_nodes_list_or_block, list):
                parent_nodes_list_or_block[:] = [n for n in parent_nodes_list_or_block if differ_instance_for_helpers._get_node_identifier(n) != child_id_in_parent]
                parent_nodes_list_or_block.append(chosen_node)
            else:
                if not parent_nodes_list_or_block.replace_child(child_id_in_parent, chosen_node):
                    parent_nodes_list_or_block.add_child_at_appropriate_location(chosen_node)
        elif change_obj.type in ['MOD_DELETED', 'VANILLA_DELETED', 'MOD_DELETED_VANILLA_ALSO_DELETED', 'CONFLICT_DELETION']:
            # For simulation, auto-delete
            if isinstance(parent_nodes_list_or_block, list):
                parent_nodes_list_or_block[:] = [n for n in parent_nodes_list_or_block if differ_instance_for_helpers._get_node_identifier(n) != child_id_in_parent]
            else:
                parent_nodes_list_or_block.remove_child(child_id_in_parent)

    for change in changes:
        _simulate_apply_change(simulated_merged_nodes, change, differ)

    simulated_merged_content = PdsParser._nodes_to_string(simulated_merged_nodes)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_SIMULATED_MERGED.txt")), simulated_merged_content)
    print(f"  Simulated merged content saved to: {os.path.basename(new_filepath).replace('.txt', '_SIMULATED_MERGED.txt')}")
    
    print(f"\n--- DIFF: SIMULATED MERGED vs NEW VANILLA RAW for '{test_name}' (Output to _MERGED_VS_NEW_RAW.diff) ---")
    diff_filename = os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_MERGED_VS_NEW_RAW.diff"))
    with open(diff_filename, 'w', encoding='utf-8') as f_diff:
        diff_lines = list(difflib.unified_diff(
            (new_content_raw if new_content_raw else "").strip().splitlines(keepends=True),
            simulated_merged_content.strip().splitlines(keepends=True),
            fromfile='NEW_VANILLA_RAW', tofile='SIMULATED_MERGED_OUTPUT', lineterm=''
        ))
        if diff_lines:
            f_diff.writelines(diff_lines)
            print(f"  Diff saved to: {os.path.basename(diff_filename)}")
        else:
            print("  SIMULATED MERGED content is identical to RAW NEW VANILLA (no meaningful changes applied by merge logic in this simulation).")
    print("-" * 80)


# --- Run Tests with Your Real Files ---
print("Running diff tests with real CK3 files.")

# Test 1: siege_events.txt
run_and_print_diff("Siege Events (real files)", 
                   SIEGE_EVENTS_OLD_VANILLA_PATH, 
                   SIEGE_EVENTS_MOD_PATH, 
                   SIEGE_EVENTS_NEW_VANILLA_PATH)

# Test 2: 00_tribal_innovations.txt
run_and_print_diff("00_tribal_innovations (real files)",
                   INNOVATIONS_OLD_VANILLA_PATH,
                   INNOVATIONS_MOD_PATH,
                   INNOVATIONS_NEW_VANILLA_PATH)

print("\n--- Real File Diff Tests Complete ---")

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--------------------------------------------------------------------------------
Running diff tests with real CK3 files.

==================== RUNNING DIFF TEST: Siege Events (real files) ====================
Output files for this test will be saved to: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\test_output\Siege_Events_real_files\20250526_163245
  Raw files saved: _OLD_RAW.txt, _MOD_RAW.txt, _NEW_RAW.txt
  Reconstructed files saved: _OLD_RECONSTRUCTED.txt, _MOD_RECONSTRUCTED.txt, _NEW_RECONSTRUCTED.txt




--- DETECTED CHANGES for 'Siege Events (real files)' (12 changes) ---
PdsChange(Type='VANILLA_DELETED                    ', Path='siege.0002 =.after =', Parent='siege.0002 =', 
          Nodes=[O:after =={...}, M:after =={...}, N:ABSENT])
PdsChange(Type='VANILLA_ADDED                      ', Path='siege.0002 =.immediate =.remove_variable', Parent='siege.0002 =.imm

In [1]:
import os
import sys
import difflib
from datetime import datetime
import re  # Make sure this import is present!

# Add the current directory to Python path to ensure local imports
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# IMPORTANT: Ensure your pds_parser.py, pds_differ.py, and pds_postprocessor.py are the LATEST versions.
# You MUST restart your Jupyter Notebook kernel before running this cell if you've changed those files!
# Note: PdsPostProcessor is only used in the dedicated fidelity test below, not the main diff flow.
from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsList, PdsComment, PdsOperatorCondition 
from pds_differ import PdsDiffer, PdsChange
# Keep import for the dedicated test function below, even if not used in main diff
from pds_postprocessor import PdsPostProcessor 

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print(f"--- Successfully imported PdsPostProcessor from: {PdsPostProcessor.__module__}.py ---") 
print("-" * 80)

# --- Define Paths to Your Real CK3 Files ---
# IMPORTANT: Replace these with the actual paths on your system
# Ensure these files exist and represent your mod, old vanilla, and new vanilla states.

SIEGE_EVENTS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\events\siege_events.txt"
SIEGE_EVENTS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\events\siege_events.txt"
SIEGE_EVENTS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\events\siege_events.txt"

INNOVATIONS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\common\culture\innovations\00_tribal_innovations.txt"


# Helper function to read file content
def get_file_content(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f:
            return f.read()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception as e_inner:
            print(f"ERROR reading {filepath} with utf-8 fallback: {e_inner}", file=sys.stderr)
            return None
    except FileNotFoundError:
        print(f"WARNING: File not found at {filepath}", file=sys.stderr)
        return None
    except Exception as e:
        print(f"ERROR reading {filepath}: {e}", file=sys.stderr)
        return None

# Helper to write content to a file
def write_to_file(filepath, content):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    try:
        with open(filepath, 'w', encoding='utf-8-sig') as f:
            f.write(content)
        return True
    except Exception as e:
        print(f"ERROR writing to {filepath}: {e}", file=sys.stderr)
        return False

# --- Helper Functions for Tree Manipulation (Copied from mod_updater.py) ---
# These are needed for the simulated merge in the test script.
# Assuming find_node_by_path and get_parent_node_by_path are defined here or accessible
# If they are methods of PdsBlock in pds_parser.py, you might not need these here
# but keeping them here for completeness as per previous scripts.
def find_node_by_path(root_nodes_list, key_path, differ_util): 
    if not key_path: return None
    current_nodes_to_search = root_nodes_list 

    for i, segment in enumerate(key_path):
        found_at_current_level = False
        for node in current_nodes_to_search: 
            # Use differ_util._get_node_identifier to handle key comparisons consistently
            node_identifier = differ_util._get_node_identifier(node)
            if node_identifier == segment:
                if i == len(key_path) - 1: 
                    return node 
                elif isinstance(node, PdsBlock): 
                    # Assumes PdsBlock has a 'children' attribute
                    current_nodes_to_search = node.children 
                    found_at_current_level = True
                    break 
                else: return None 
        if not found_at_current_level: return None 
    return None 

def get_parent_node_by_path(root_nodes_list, key_path, differ_util): 
    if not key_path or len(key_path) < 1: return None, None
    if len(key_path) == 1: # Top-level node, its "parent" is the root_nodes_list itself
        return root_nodes_list, key_path[0] 

    parent_path = key_path[:-1] 
    child_identifier = key_path[-1]

    parent_block = find_node_by_path(root_nodes_list, parent_path, differ_util)
    
    if isinstance(parent_block, PdsBlock): 
        return parent_block, child_identifier
    return None, None 


# Helper to normalize content for comparison purposes (e.g., when comparing raw vs reconstructed)
# This accounts for the parser's canonical output style (like blank lines)
# and should match what the parser's to_string produces for those elements.
def normalize_for_comparison(text):
    if not text: return ""
    # This normalization should match the *parser's* standard output behavior.
    # If the parser converts any sequence of >=2 newlines to \n\n, this should match that.
    # If the parser preserves single newlines between statements/comments, this should handle that.
    # Based on the current parser: it keeps single \n as separators, and converts >=2 \n to a PdsBlankLine which prints as \n.
    # So, normalization should replace >=2 \n with a single \n
    # A simpler normalization might be replacing any sequence of spaces/newlines with a single space, except for line breaks?
    # Let's stick to replacing >=2 newlines with one for now, which is a common canonical form.
    # This regex replaces one or more newlines followed by one or more potential newlines with whitespace with a single newline.
    normalized = re.sub(r'\n(\s*\n)+', '\n', text)
    # Also strip any leading/trailing whitespace including newlines
    return normalized.strip()

# Helper Function to Run Diff (updated to output files)
def run_and_print_diff(test_name, old_filepath, mod_filepath, new_filepath):
    print(f"\n{'='*20} RUNNING DIFF TEST: {test_name} {'='*20}")
    
    # Define output directory for this test run (Consistent Directory)
    test_output_dir = os.path.join(os.getcwd(), "test_output", test_name.replace(" ", "_").replace("(", "").replace(")", ""))
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir} (overwritten on subsequent runs)")

    old_content_raw = get_file_content(old_filepath)
    mod_content_raw = get_file_content(mod_filepath)
    new_content_raw = get_file_content(new_filepath)

    # Save raw files to output for reference
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RAW.txt")), old_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RAW.txt")), mod_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RAW.txt")), new_content_raw or "")
    print(f"  Raw files saved: _OLD_RAW.txt, _MOD_RAW.txt, _NEW_RAW.txt")

    # Parse files
    parser = PdsParser()
    old_nodes = parser.parse_file(old_filepath)
    mod_nodes = parser.parse_file(mod_filepath)
    new_nodes = parser.parse_file(new_filepath)

    # Ensure empty lists if files were not found/parsed for diffing
    if old_nodes is None: old_nodes = []
    if mod_nodes is None: mod_nodes = []
    if new_nodes is None: new_nodes = []

    if not old_nodes and not mod_nodes and not new_nodes:
        print(f"SKIPPING: No content to parse for test '{test_name}' from any source (O, M, N). Check file paths.")
        return

    # Reconstruct parsed content to verify parser (DO NOT apply post-processor here)
    reconstructed_old = PdsParser._nodes_to_string(old_nodes)
    reconstructed_mod = PdsParser._nodes_to_string(mod_nodes)
    reconstructed_new = PdsParser._nodes_to_string(new_nodes)

    # Save reconstructed files
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RECONSTRUCTED.txt")), reconstructed_old)
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RECONSTRUCTED.txt")), reconstructed_mod)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RECONSTRUCTED.txt")), reconstructed_new)
    print(f"  Reconstructed files saved (Parser output): _OLD_RECONSTRUCTED.txt, _MOD_RECONSTRUCTED.txt, _NEW_RECONSTRUCTED.txt")

    # Check for reconstruction consistency (after normalizing raw content for comparison)
    # Normalize raw content to match expected parser output formatting (e.g., blank lines)
    normalized_old_raw = normalize_for_comparison(old_content_raw)
    normalized_mod_raw = normalize_for_comparison(mod_content_raw)
    normalized_new_raw = normalize_for_comparison(new_content_raw)

    # Compare reconstructed output (straight from parser) against normalized raw
    if reconstructed_old.strip() != normalized_old_raw:
        print(f"\nWARNING: Reconstruction mismatch for OLD file: {os.path.basename(old_filepath)}. Parser output might differ slightly in whitespace. Compare _OLD_RAW.txt and _OLD_RECONSTRUCTED.txt using a diff tool (or check _OLD_RECONSTRUCTED_VS_NORMALIZED_RAW.diff).")
        with open(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RECONSTRUCTED_VS_NORMALIZED_RAW.diff")), 'w', encoding='utf-8') as f_diff:
            diff_lines = list(difflib.unified_diff(
                normalized_old_raw.splitlines(keepends=True),
                reconstructed_old.strip().splitlines(keepends=True),
                fromfile='NORMALIZED_RAW', tofile='RECONSTRUCTED_PARSER', lineterm=''
            ))
            f_diff.writelines(diff_lines)

    if reconstructed_mod.strip() != normalized_mod_raw:
        print(f"\nWARNING: Reconstruction mismatch for MOD file: {os.path.basename(mod_filepath)}. Parser output might differ slightly in whitespace. Compare _MOD_RAW.txt and _MOD_RECONSTRUCTED.txt using a diff tool (or check _MOD_RECONSTRUCTED_VS_NORMALIZED_RAW.diff).")
        with open(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RECONSTRUCTED_VS_NORMALIZED_RAW.diff")), 'w', encoding='utf-8') as f_diff:
            diff_lines = list(difflib.unified_diff(
                normalized_mod_raw.splitlines(keepends=True),
                reconstructed_mod.strip().splitlines(keepends=True),
                fromfile='NORMALIZED_RAW', tofile='RECONSTRUCTED_PARSER', lineterm=''
            ))
            f_diff.writelines(diff_lines)

    if reconstructed_new.strip() != normalized_new_raw:
        print(f"\nWARNING: Reconstruction mismatch for NEW file: {os.path.basename(new_filepath)}. Parser output might differ slightly in whitespace. Compare _NEW_RAW.txt and _NEW_RECONSTRUCTED.txt using a diff tool (or check _NEW_RECONSTRUCTED_VS_NORMALIZED_RAW.diff).")
        with open(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RECONSTRUCTED_VS_NORMALIZED_RAW.diff")), 'w', encoding='utf-8') as f_diff:
            diff_lines = list(difflib.unified_diff(
                normalized_new_raw.splitlines(keepends=True),
                reconstructed_new.strip().splitlines(keepends=True),
                fromfile='NORMALIZED_RAW', tofile='RECONSTRUCTED_PARSER', lineterm=''
            ))
            f_diff.writelines(diff_lines)

    # Run the differ (on AST nodes)
    differ = PdsDiffer() # Assuming PdsDiffer is defined elsewhere
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes) # Diff is still on AST nodes, not strings

    print(f"\n--- DETECTED CHANGES for '{test_name}' ({len(changes)} changes) ---")
    if not changes:
        print("    No significant changes detected (or only identical comments/blank lines were filtered).")
    for change in changes:
        print(change) # PdsChange.__repr__ provides detailed formatting
    
    # --- SIMULATE MERGE (for display purposes only in this test script) ---
    # Deep copy new_nodes for modification. The merge logic should correctly set indent_levels.
    simulated_merged_nodes = [node.copy() for node in new_nodes] 
    
    # Pass simulated_merged_nodes as the target list
    # This helper applies changes (add/modify/delete) and ensures indent_levels are set.
    def _simulate_apply_change(target_nodes_list_root, change_obj, differ_instance_for_helpers):
        parent_nodes_list_or_block, child_id_in_parent = get_parent_node_by_path(target_nodes_list_root, change_obj.key_path, differ_instance_for_helpers)
        
        if parent_nodes_list_or_block is None:
            # This happens if a node's parent in O/M doesn't exist in N.
            # For simulation, we can't apply changes to non-existent parents.
            # In a real merge, this would require more complex parent insertion logic.
            print(f"  SIMULATED MERGE WARNING: Parent for '{'.'.join(change_obj.key_path)}' not found in target tree. Cannot apply change.")
            return

        # Optional: Add a comment indicating the merge action
        sim_comment = f"SimulatedMerge:{change_obj.type}" # Simpler comment for diff readability

        if change_obj.type == 'MOD_ADDED':
            new_node = change_obj.mod_node.copy()
            if hasattr(new_node, 'comment_text_on_line') and new_node.comment_text_on_line is not None: 
                 new_node.comment_text_on_line += f" {sim_comment}"
            else: 
                 new_node.comment_text_on_line = sim_comment
            
            if isinstance(parent_nodes_list_or_block, list): # Root level addition
                new_node.indent_level = 0 # Set indent for root node
                parent_nodes_list_or_block.append(new_node)
            else: # Nested addition (PdsBlock instance)
                # add_child_at_appropriate_location sets indent_level
                parent_nodes_list_or_block.add_child_at_appropriate_location(new_node) 
        
        elif change_obj.type == 'MOD_MODIFIED':
            new_node = change_obj.mod_node.copy()
            if hasattr(new_node, 'comment_text_on_line') and new_node.comment_text_on_line is not None: 
                 new_node.comment_text_on_line += f" {sim_comment}"
            else: 
                 new_node.comment_text_on_line = sim_comment

            if isinstance(parent_nodes_list_or_block, list): # Root level modification
                new_node.indent_level = 0 # Set indent for root node
                # Replace node in the root list
                for idx, node in enumerate(parent_nodes_list_or_block):
                    if differ_instance_for_helpers._get_node_identifier(node) == child_id_in_parent:
                        parent_nodes_list_or_block[idx] = new_node
                        break
            else: # Nested modification (PdsBlock instance)
                 # replace_child sets indent_level
                parent_nodes_list_or_block.replace_child(child_id_in_parent, new_node)
        
        elif change_obj.type == 'VANILLA_ADDED' or change_obj.type == 'VANILLA_MODIFIED':
            # These changes are already in the target (simulated_merged_nodes starts as NEW_VANILLA).
            # We might add a comment to note the change in the merged output.
            node_in_output = find_node_by_path(target_nodes_list_root, change_obj.key_path, differ_instance_for_helpers)
            if node_in_output:
                 if hasattr(node_in_output, 'comment_text_on_line') and node_in_output.comment_text_on_line is not None:
                    node_in_output.comment_text_on_line += f" {sim_comment}"
                 else: node_in_output.comment_text_on_line = sim_comment

        elif change_obj.type == 'CONFLICT_MODIFIED' or change_obj.type == 'CONFLICT_ADDITION':
            # For simulation, auto-choose MOD's version and mark it.
            chosen_node = change_obj.mod_node.copy() 
            if hasattr(chosen_node, 'comment_text_on_line') and chosen_node.comment_text_on_line is not None: 
                 chosen_node.comment_text_on_line += f" {sim_comment}_MOD_CHOSEN"
            else: 
                 chosen_node.comment_text_on_line = f"{sim_comment}_MOD_CHOSEN"

            if isinstance(parent_nodes_list_or_block, list): # Root level
                chosen_node.indent_level = 0 # Set indent for root node
                # Remove old node (if exists) and add new node
                parent_nodes_list_or_block[:] = [n for n in parent_nodes_list_or_block if differ_instance_for_helpers._get_node_identifier(n) != child_id_in_parent]
                parent_nodes_list_or_block.append(chosen_node)
            else: # Nested
                # replace_child/add_child_at_appropriate_location set indent_level
                if not parent_nodes_list_or_block.replace_child(child_id_in_parent, chosen_node):
                    parent_nodes_list_or_block.add_child_at_appropriate_location(chosen_node)

        elif change_obj.type in ['MOD_DELETED', 'VANILLA_DELETED', 'MOD_DELETED_VANILLA_ALSO_DELETED', 'CONFLICT_DELETION']:
            # For simulation, auto-delete
            if isinstance(parent_nodes_list_or_block, list): # Root level
                parent_nodes_list_or_block[:] = [n for n in parent_nodes_list_or_block if differ_instance_for_helpers._get_node_identifier(n) != child_id_in_parent]
            else: # Nested
                parent_nodes_list_or_block.remove_child(child_id_in_parent)


    for change in changes:
        _simulate_apply_change(simulated_merged_nodes, change, differ)

    # Convert simulated merged AST to string (NO POST-PROCESSING APPLIED HERE)
    # The formatting comes directly from the parser's to_string methods.
    simulated_merged_content = PdsParser._nodes_to_string(simulated_merged_nodes)
    
    # !!! IMPORTANT: Post-processing is NOT applied to simulated_merged_content here
    # if you want to rely solely on the parser's output formatting.
    # If you DID want minimal post-processing (e.g., normalizing blank lines), you'd apply it here.
    # Example minimal post-processing (if you decide you need *some* cleanup):
    # minimal_post_processor = PdsPostProcessor() # Assuming PdsPostProcessor class exists and has process method
    # simulated_merged_content = minimal_post_processor.process(simulated_merged_content)


    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_SIMULATED_MERGED.txt")), simulated_merged_content)
    print(f"  Simulated merged content saved to: {os.path.basename(new_filepath).replace('.txt', '_SIMULATED_MERGED.txt')}")
    
    print(f"\n--- DIFF: SIMULATED MERGED (Parser Output) vs NORMALIZED NEW VANILLA RAW for '{test_name}' ---")
    diff_filename = os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_MERGED_VS_NEW_RAW.diff"))
    with open(diff_filename, 'w', encoding='utf-8') as f_diff:
        # Compare simulated merged content (parser output) against normalized new vanilla
        diff_lines = list(difflib.unified_diff(
            (normalize_for_comparison(new_content_raw) if new_content_raw else "").splitlines(keepends=True),
            simulated_merged_content.strip().splitlines(keepends=True),
            fromfile='NORMALIZED_NEW_VANILLA_RAW', tofile='SIMULATED_MERGED_PARSER_OUTPUT', lineterm=''
        ))
        if diff_lines:
            f_diff.writelines(diff_lines)
            print(f"  Diff saved to: {os.path.basename(diff_filename)}")
        else:
            print("  SIMULATED MERGED content (parser output) is identical to NORMALIZED NEW VANILLA.")
    print("-" * 80)


# --- Dedicated Post-Processor Fidelity Tests ---
# Keep this section to test the post-processor in isolation if needed later.
def test_post_processor_fidelity(test_name, input_filepath, expected_output_filepath=None):
    """
    Tests the PdsPostProcessor in isolation by parsing a file, reconstructing it,
    and then applying the post-processor, comparing the result to an expected output.
    If no explicit expected_output_filepath is provided, it compares against a normalized
    version of the raw input.
    """
    print(f"\n{'='*20} RUNNING POST-PROCESSOR FIDELITY TEST: {test_name} {'='*20}")
    
    # Define output directory for this test run (Consistent Directory)
    test_output_dir = os.path.join(os.getcwd(), "test_output", f"PostProcessorTest_{test_name.replace(' ', '_')}")
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir} (overwritten on subsequent runs)")

    raw_input_content = get_file_content(input_filepath)
    if raw_input_content is None:
        print(f"SKIPPING: Input file not found for post-processor test: {input_filepath}")
        return

    write_to_file(os.path.join(test_output_dir, os.path.basename(input_filepath).replace(".txt", "_RAW_INPUT.txt")), raw_input_content)
    print(f"  Raw input content saved to: {os.path.basename(input_filepath).replace('.txt', '_RAW_INPUT.txt')}")

    # 1. Parse the raw input (this will produce the parser's standard output with its formatting)
    parser = PdsParser()
    parsed_nodes = parser.parse_file(input_filepath)
    parser_reconstructed_content = PdsParser._nodes_to_string(parsed_nodes)
    write_to_file(os.path.join(test_output_dir, os.path.basename(input_filepath).replace(".txt", "_PARSED_RECONSTRUCTED.txt")), parser_reconstructed_content)
    print(f"  Parsed and reconstructed content saved to: {os.path.basename(input_filepath).replace('.txt', '_PARSED_RECONSTRUCTED.txt')}")

    # 2. Apply the post-processor to the parser's output
    post_processor = PdsPostProcessor() # PdsPostProcessor is used here
    post_processed_content = post_processor.process(parser_reconstructed_content)
    write_to_file(os.path.join(test_output_dir, os.path.basename(input_filepath).replace(".txt", "_POST_PROCESSED.txt")), post_processed_content)
    print(f"  Post-processed content saved to: {os.path.basename(input_filepath).replace('.txt', '_POST_PROCESSED.txt')}")

    # 3. Determine comparison target
    comparison_target_content = None
    target_label = ""
    if expected_output_filepath and os.path.exists(expected_output_filepath):
        comparison_target_content = get_file_content(expected_output_filepath)
        target_label = "EXPLICIT_EXPECTED_OUTPUT"
        write_to_file(os.path.join(test_output_dir, os.path.basename(expected_output_filepath).replace(".txt", "_EXPECTED_OUTPUT.txt")), comparison_target_content)
        print(f"  Comparing against explicit expected output: {os.path.basename(expected_output_filepath)}")
    else:
        # If no explicit expected output, compare against a normalized version of the original raw input
        # This normalization should match the *intended* final output formatting, which might
        # be different from the parser's raw output if the post-processor fixes things like blank lines.
        # If post-processor handles blank lines, use normalize_for_comparison that reduces >=3 to \n\n
        # If parser handles blank lines perfectly, maybe just strip() is enough for normalization?
        # Let's assume the parser does blank lines well and this normalization should remove multi-blank lines and strip.
        comparison_target_content = normalize_for_comparison(raw_input_content) # Use the normalize_for_comparison helper
        target_label = "NORMALIZED_RAW_INPUT"
        write_to_file(os.path.join(test_output_dir, os.path.basename(input_filepath).replace(".txt", "_NORMALIZED_RAW_INPUT.txt")), comparison_target_content)
        print(f"  No explicit expected output, comparing against normalized raw input.")


    if comparison_target_content is None:
        print(f"  SKIPPING COMPARISON: No valid content for comparison target.")
        return

    diff_filename = os.path.join(test_output_dir, "post_processed_vs_target.diff")
    with open(diff_filename, 'w', encoding='utf-8') as f_diff:
        diff_lines = list(difflib.unified_diff(
            (comparison_target_content if comparison_target_content else "").strip().splitlines(keepends=True),
            post_processed_content.strip().splitlines(keepends=True),
            fromfile=target_label, tofile='POST_PROCESSED_OUTPUT', lineterm=''
        ))
        if diff_lines:
            f_diff.writelines(diff_lines)
            print(f"  Diff saved to: {os.path.basename(diff_filename)}")
        else:
            print("  POST-PROCESSED content is identical to the comparison target (perfect formatting).")
    print("-" * 80)


# --- Run Tests with Your Real Files ---
print("Running diff tests with real CK3 files.")

# Test 1: siege_events.txt
run_and_print_diff("Siege Events (real files)", 
                   SIEGE_EVENTS_OLD_VANILLA_PATH, 
                   SIEGE_EVENTS_MOD_PATH, 
                   SIEGE_EVENTS_NEW_VANILLA_PATH)

# Test 2: 00_tribal_innovations.txt
run_and_print_diff("00_tribal_innovations (real files)",
                   INNOVATIONS_OLD_VANILLA_PATH,
                   INNOVATIONS_MOD_PATH,
                   INNOVATIONS_NEW_VANILLA_PATH)

# --- Dedicated Post-Processor Fidelity Tests ---
# Use one of your actual files as input. If you have an ideal formatted version,
# you can provide it as expected_output_filepath. Otherwise, it compares against
# a version of the raw input with its blank lines normalized.
print("\nRunning dedicated Post-Processor Fidelity Tests.")
# Keep these tests even if you rely primarily on the parser, they help validate
# the post-processor if you ever decide to use it for minor cleanup.
test_post_processor_fidelity("Siege Events Post-Processing Test",
                             SIEGE_EVENTS_NEW_VANILLA_PATH,
                             # Optional: r"C:\path\to\your\ideal_siege_events_formatted.txt"
                            )

test_post_processor_fidelity("Innovations Post-Processing Test",
                             INNOVATIONS_NEW_VANILLA_PATH,
                             # Optional: r"C:\path\to\your\ideal_innovations_formatted.txt"
                            )

print("\n--- All Tests Complete ---")

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--- Successfully imported PdsPostProcessor from: pds_postprocessor.py ---
--------------------------------------------------------------------------------
Running diff tests with real CK3 files.

==================== RUNNING DIFF TEST: Siege Events (real files) ====================
Output files for this test will be saved to: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\test_output\Siege_Events_real_files (overwritten on subsequent runs)
  Raw files saved: _OLD_RAW.txt, _MOD_RAW.txt, _NEW_RAW.txt
  Reconstructed files saved (Parser output): _OLD_RECONSTRUCTED.txt, _MOD_RECONSTRUCTED.txt, _NEW_RECONSTRUCTED.txt




--- DETECTED CHANGES for 'Siege Events (real files)' (12 changes) ---
PdsChange(Type='VANILLA_DELETED                    ', Path='siege.0002 =.after =', Parent='siege.0002 =', 
          Nodes=[O:after =={...}, M:after =={...}, N:ABSENT])
PdsChange(Type='VANI

In [ ]:
import os
import sys
import difflib
from datetime import datetime
import re
import collections # Using deque for recursive search

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsList, PdsComment, PdsOperatorCondition, PdsNode
from pds_differ import PdsDiffer, PdsChange # Assuming PdsDiffer is the revised version

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print("-" * 80)

# --- Paths (UNCHANGED) ---
SIEGE_EVENTS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\events\siege_events.txt"
SIEGE_EVENTS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\events\siege_events.txt"
SIEGE_EVENTS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\events\siege_events.txt"
INNOVATIONS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\common\culture\innovations\00_tribal_innovations.txt"

# --- File I/O Helpers (UNCHANGED) ---
def get_file_content(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f: return f.read()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='utf-8') as f: return f.read()
        except Exception as e_inner: print(f"ERROR reading {filepath} (fallback): {e_inner}", file=sys.stderr); return None
    except FileNotFoundError: print(f"WARNING: File not found: {filepath}", file=sys.stderr); return None
    except Exception as e: print(f"ERROR reading {filepath}: {e}", file=sys.stderr); return None

def write_to_file(filepath, content):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    try:
        with open(filepath, 'w', encoding='utf-8-sig') as f: f.write(content); return True
    except Exception as e: print(f"ERROR writing to {filepath}: {e}", file=sys.stderr); return False

# --- Tree Navigation Helpers ---

def _parse_indexed_identifier_for_nav(identifier_full):
    """Helper to parse 'key___index' or just 'key' for navigation utilities."""
    if isinstance(identifier_full, str) and "___" in identifier_full:
        parts = identifier_full.split("___", 1)
        base_key = parts[0]
        try:
            index = int(parts[1])
            return base_key, index, True # is_indexed
        except ValueError: # "___" was part of the key itself
            return identifier_full, 0, False # Treat as non-indexed key
    return identifier_full, 0, False # Non-indexed

def find_node_by_path(root_nodes_list_or_block, key_path_list):
    """
    Finds a node by its path (list of segments).
    root_nodes_list_or_block can be the root list or a PdsBlock.
    Returns the node if found, None otherwise.
    """
    if not key_path_list: return root_nodes_list_or_block # Empty path might mean the root itself? Let's allow this.

    current_container = root_nodes_list_or_block
    target_node = None

    # Handle starting from a single block vs a list of root nodes
    is_starting_from_list = isinstance(current_container, list)

    path_iter = iter(key_path_list)

    # Handle the first segment differently if starting from a list
    if is_starting_from_list:
        first_segment_full = next(path_iter, None)
        if first_segment_full is None: return current_container # Path was just []
        
        segment_key_part, target_idx_in_group, is_indexed = _parse_indexed_identifier_for_nav(first_segment_full)

        current_key_group_count = 0
        found_in_segment = False
        for root_node in current_container:
            # Use PdsDiffer's identifier logic for matching
            root_node_raw_id = PdsDiffer()._get_raw_identifier_for_node(root_node)
            if root_node_raw_id == segment_key_part:
                if not is_indexed or current_key_group_count == target_idx_in_group:
                    target_node = root_node
                    found_in_segment = True
                    break
                current_key_group_count += 1
        if not found_in_segment: return None # First segment not found in root list
        
        # The found target_node is now the container for the next step
        current_container = target_node
    # Else: Starting from a PdsBlock, handle all segments iteratively

    # Process remaining segments (or all segments if starting from a PdsBlock)
    for segment_full in path_iter:
        if not isinstance(current_container, PdsBlock):
            return None # Cannot navigate children if current node is not a block

        segment_key_part, target_idx_in_group, is_indexed = _parse_indexed_identifier_for_nav(segment_full)

        current_key_group_count = 0
        found_in_segment = False
        target_node = None # Reset target for this segment
        for child_node in current_container.children:
            child_raw_id = PdsDiffer()._get_raw_identifier_for_node(child_node)
            if child_raw_id == segment_key_part:
                if not is_indexed or current_key_group_count == target_idx_in_group:
                    target_node = child_node
                    found_in_segment = True
                    break
                current_key_group_count += 1

        if not found_in_segment: return None # Segment not found in current block's children
        current_container = target_node # Set target for next iteration

    # After iterating through all segments, current_container is the final target node
    return current_container


def find_parent_node_by_path(root_nodes_list, key_path_list):
    """
    Finds the parent container for the node specified by key_path_list.
    Returns: parent_container (root_nodes_list or a PdsBlock), or None if parent not found.
    """
    if not key_path_list:
        return None # Cannot find parent of root itself

    parent_path = key_path_list[:-1]
    
    if not parent_path: # Target is a root-level node, parent is the root list
        return root_nodes_list

    # Target is nested, parent must be a PdsBlock
    parent_node = find_node_by_path(root_nodes_list, parent_path)

    if isinstance(parent_node, PdsBlock):
        return parent_node
    else:
        return None # Parent path didn't lead to a PdsBlock


def find_copied_node_recursive(sim_tree_root_list, original_new_node: PdsNode):
    """
    Recursively searches the simulated tree (copy of New AST) for a node
    that is structurally equal to the original_new_node instance.
    Returns the copied node instance in the simulated tree, or None.
    """
    if original_new_node is None: return None

    queue = collections.deque(sim_tree_root_list)

    while queue:
        current_node = queue.popleft()

        # Check if this node is a copy of the original new node
        # Use deep comparison (shallow_block_comparison=False)
        if current_node is not None and PdsDiffer()._are_nodes_structurally_equal(current_node, original_new_node, shallow_block_comparison=False):
            return current_node # Found the copy

        # Add children to the queue if it's a block
        if isinstance(current_node, PdsBlock):
            queue.extend(current_node.children)
        # Note: PdsKeyValuePair/PdsOperatorCondition can have PdsBlock as value
        # We need to check their 'value' if it's a PdsBlock for recursion.
        # This requires inspecting specific node types if the structure is not purely Block->children.
        # For now, assuming children are directly in PdsBlock. Children of KVP/OpCondition values are handled by their to_string/copy/eq.
        # If needed, could add:
        # if isinstance(current_node, (PdsKeyValuePair, PdsOperatorCondition)) and isinstance(current_node.value, PdsBlock):
        #     queue.append(current_node.value)
    return None


def find_node_and_parent_in_sim_tree(sim_root_list, chg_obj: PdsChange):
    """
    Attempts to find the target node AND its parent in the simulated tree.
    Uses chg_obj.new_node (from original New AST) as the primary identifier when available.
    Falls back to path-based search for parent if chg_obj.new_node is None (e.g., for ADDs).

    Returns: (target_node_in_sim, parent_in_sim, child_key_for_parent_method)
             target_node_in_sim: The node in the simulated tree to modify/remove, or None if adding.
             parent_in_sim: The container (list or PdsBlock) in the simulated tree.
             child_key_for_parent_method: The last segment of chg_obj.key_path, used by parent methods like remove_child/replace_child.
    """
    target_node_in_sim = None
    parent_in_sim = None
    child_key_for_parent_method = chg_obj.key_path[-1] if chg_obj.key_path else None

    if chg_obj.new_node is not None:
        # Strategy 1: Find the copy of the original New node in the simulation tree.
        # This is the most robust method if the item existed in New Vanilla.
        target_node_in_sim = find_copied_node_recursive(sim_root_list, chg_obj.new_node)

        if target_node_in_sim:
             # If found, we need its parent. This is tricky without tracking parents during recursion.
             # As a fallback, try finding the parent using the context_parent_path on the simulated tree.
             # This is still not ideal if Vanilla reordered, but better than using the full path.
             parent_in_sim = find_parent_node_by_path(sim_root_list, chg_obj.key_path)
             # Verify the found parent actually contains the target_node_in_sim instance
             if parent_in_sim is None: # Target was root-level
                 if target_node_in_sim in sim_root_list:
                     parent_in_sim = sim_root_list
                 else:
                     print(f"  SIM MERGE DEBUG: Found target node by copy, but it's not in root list {child_key_for_parent_method}.")
                     return None, None, None # Should not happen if target was root
             elif isinstance(parent_in_sim, PdsBlock): # Target was nested in a block
                  if target_node_in_sim not in parent_in_sim.children:
                      print(f"  SIM MERGE DEBUG: Found target node by copy, but it's not a child of parent found by path {'.'.join(chg_obj.context_parent_path)}.")
                      # This indicates the path-based parent lookup is unreliable.
                      # A more complex parent tracking during recursive search would be needed here.
                      # For now, we must fail if the path-based parent is wrong.
                      return None, None, None
        else:
             print(f"  SIM MERGE DEBUG: Target node corresponding to new_node (type {type(chg_obj.new_node).__name__}, key {getattr(chg_obj.new_node, 'key', 'N/A')}) not found in sim tree via structural match for path {'.'.join(chg_obj.key_path)}.")
             # If new_node existed but its copy isn't found, something is wrong or it was implicitly deleted/replaced.
             return None, None, None

    else: # chg_obj.new_node is None (item was ADDED by Mod, or DELETED by Vanilla/Both)
        # Strategy 2: Find the parent node using the context_parent_path on the simulated tree.
        parent_in_sim = find_parent_node_by_path(sim_root_list, chg_obj.key_path) # Use full path to get parent

        if parent_in_sim is None:
            # If path is just one segment (root level), parent_path is [], find_parent_node_by_path should return sim_root_list
            if chg_obj.key_path and len(chg_obj.key_path) == 1:
                 parent_in_sim = sim_root_list # Target is root-level, parent is the list
            else:
                 print(f"  SIM MERGE DEBUG: Parent for path '{'.'.join(chg_obj.context_parent_path)}' not found in sim tree.")
                 return None, None, None # Parent not found by path

        # For ADDED changes, target_node_in_sim remains None.
        # For DELETED changes, target_node_in_sim should correspond to the node that *was* in New Vanilla,
        # but since new_node is None, it means Vanilla already deleted it. So target_node_in_sim remains None.
        # The operation (remove/add) will be relative to parent_in_sim using child_key_for_parent_method or insertion logic.


    # If we reached here, parent_in_sim should be valid (list or PdsBlock)
    if not isinstance(parent_in_sim, (list, PdsBlock)):
         print(f"  SIM MERGE ERROR: Parent found is not list or PdsBlock (is {type(parent_in_sim)}) for path {'.'.join(chg_obj.key_path)}.")
         return None, None, None


    return target_node_in_sim, parent_in_sim, child_key_for_parent_method


def normalize_for_comparison(text): # UNCHANGED
    if not text: return ""
    normalized = re.sub(r'\n(\s*\n)+', '\n', text)
    return normalized.strip()

def run_and_print_diff(test_name, old_filepath, mod_filepath, new_filepath):
    print(f"\n{'='*20} RUNNING DIFF TEST: {test_name} {'='*20}")
    test_output_dir = os.path.join(os.getcwd(), "test_output", test_name.replace(" ", "_").replace("(", "").replace(")", ""))
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir}")

    # --- File Parsing and AST generation ---
    old_content_raw = get_file_content(old_filepath)
    mod_content_raw = get_file_content(mod_filepath)
    new_content_raw = get_file_content(new_filepath)
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RAW.txt")), old_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RAW.txt")), mod_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RAW.txt")), new_content_raw or "")
    print(f"  Raw files saved.")

    parser = PdsParser()
    old_nodes = parser.parse_file(old_filepath)
    mod_nodes = parser.parse_file(mod_filepath)
    new_nodes = parser.parse_file(new_filepath)

    if old_nodes is None: old_nodes = []
    if mod_nodes is None: mod_nodes = []
    if new_nodes is None: new_nodes = []

    if not old_nodes and not mod_nodes and not new_nodes and not (old_content_raw or mod_content_raw or new_content_raw):
        print(f"SKIPPING: No content for test '{test_name}'.")
        return

    # --- Reconstruction and Consistency Checks ---
    reconstructed_old = PdsParser._nodes_to_string(old_nodes)
    reconstructed_mod = PdsParser._nodes_to_string(mod_nodes)
    reconstructed_new = PdsParser._nodes_to_string(new_nodes)
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RECONSTRUCTED.txt")), reconstructed_old)
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RECONSTRUCTED.txt")), reconstructed_mod)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RECONSTRUCTED.txt")), reconstructed_new)
    print(f"  Reconstructed files saved.")

    normalized_old_raw = normalize_for_comparison(old_content_raw or "")
    normalized_mod_raw = normalize_for_comparison(mod_content_raw or "")
    normalized_new_raw = normalize_for_comparison(new_content_raw or "")

    # --- Reconstruction mismatch warnings (UNCHANGED) ---
    if reconstructed_old.strip() != normalized_old_raw: print(f"\nWARNING: Reconstruction mismatch for OLD file: {os.path.basename(old_filepath)}.")
    if reconstructed_mod.strip() != normalized_mod_raw: print(f"\nWARNING: Reconstruction mismatch for MOD file: {os.path.basename(mod_filepath)}.")
    if reconstructed_new.strip() != normalized_new_raw: print(f"\nWARNING: Reconstruction mismatch for NEW file: {os.path.basename(new_filepath)}.")

    # --- Diffing ---
    differ = PdsDiffer()
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes)

    print(f"\n--- DETECTED CHANGES for '{test_name}' ({len(changes)} changes) ---")
    if not changes: print("    No significant changes detected.")
    for change in changes: print(change)

    # --- Simulation ---
    simulated_merged_nodes_root_list = [node.copy() for node in new_nodes] # Start with New Vanilla

    def _simulate_apply_change(target_tree_root_list, chg_obj: PdsChange):
        print(f"\n  Attempting to apply change: {chg_obj.type} at path {'->'.join(chg_obj.key_path)}")
        sim_comment_text = f"SimMerge:{chg_obj.type}"

        def _add_comment(node, comment):
            if node and hasattr(node, 'comment_text_on_line'):
                node.comment_text_on_line = f"{node.comment_text_on_line} {comment}" if node.comment_text_on_line else comment

        # Find the target node OR its parent in the simulated tree
        target_node_in_sim, parent_in_sim, child_key_for_parent_method = find_node_and_parent_in_sim_tree(
            target_tree_root_list, chg_obj
        )

        if parent_in_sim is None:
             print(f"  SIM MERGE FAIL: Could not find parent or target node location in simulation tree for path '{'.'.join(chg_obj.key_path)}'. Skipping.")
             return

        print(f"    Parent found (Type: {type(parent_in_sim).__name__}, Path: {'->'.join(chg_obj.context_parent_path) if chg_obj.context_parent_path else 'ROOT'}), Target Node In Sim Found: {target_node_in_sim is not None}")
        # print(f"    Child key for parent method: {child_key_for_parent_method}") # Often useful for debug

        # Apply Mod-specific changes
        if chg_obj.type == 'MOD_MODIFIED':
            if target_node_in_sim and chg_obj.mod_node:
                mod_node_copy = chg_obj.mod_node.copy()
                _add_comment(mod_node_copy, sim_comment_text)
                if isinstance(parent_in_sim, list): # Target is a root node
                     try:
                        idx = parent_in_sim.index(target_node_in_sim)
                        mod_node_copy.indent_level = 0
                        parent_in_sim[idx] = mod_node_copy
                        print(f"      Applied MOD_MODIFIED: Replaced root node at index {idx}.")
                     except ValueError: print(f"    SIM MERGE ERROR (MOD_MOD Root): Target instance not found in root list for replacement.")
                elif isinstance(parent_in_sim, PdsBlock): # Target is a child in a block
                     # Need the child's key/index identifier IN THE SIM TREE's context.
                     # Since we found target_node_in_sim, we know it's a child. Find its *current* id.
                     sim_child_id = None
                     current_key_group_count = 0
                     target_raw_id = PdsDiffer()._get_raw_identifier_for_node(target_node_in_sim)
                     for idx, child in enumerate(parent_in_sim.children):
                         child_raw_id = PdsDiffer()._get_raw_identifier_for_node(child)
                         if child is target_node_in_sim: # Found the exact instance
                            sim_child_id = f"{target_raw_id}___{current_key_group_count}"
                            break
                         if child_raw_id == target_raw_id:
                             current_key_group_count += 1

                     if sim_child_id:
                         if parent_in_sim.replace_child(sim_child_id, mod_node_copy):
                             print(f"      Applied MOD_MODIFIED: Replaced child '{sim_child_id}' in block '{parent_in_sim.key}'.")
                         else:
                            print(f"    SIM MERGE WARN (MOD_MOD Block): PdsBlock.replace_child failed for '{sim_child_id}' in '{parent_in_sim.key}'.")
                     else:
                         print(f"    SIM MERGE ERROR (MOD_MOD Block): Could not determine child identifier for replacement in block '{parent_in_sim.key}'.")
                else:
                    print(f"    SIM MERGE ERROR (MOD_MOD): Parent is invalid type {type(parent_in_sim).__name__}.")
            else:
                print(f"    SIM MERGE FAIL (MOD_MODIFIED): Target node or mod node absent.")

        elif chg_obj.type == 'MOD_ADDED':
            if chg_obj.mod_node:
                mod_node_copy = chg_obj.mod_node.copy()
                _add_comment(mod_node_copy, sim_comment_text)
                if isinstance(parent_in_sim, list): # Adding to root list
                    mod_node_copy.indent_level = 0
                    parent_in_sim.append(mod_node_copy) # TODO: Use L1idx from INSERTED path for better ordering hint
                    print(f"      Applied MOD_ADDED: Appended root node.")
                elif isinstance(parent_in_sim, PdsBlock):
                    # add_child_at_appropriate_location can use target_sibling_identifier_full for hint
                    # The last segment of the OLD-centric key_path (child_key_for_parent_method)
                    # is the INSERTED string. It has L1idx which is index in Old.
                    # It's hard to map this to an index in the New-based parent children list.
                    # Just adding at the end for now, or use the block's default sorting.
                    parent_in_sim.add_child_at_appropriate_location(mod_node_copy) # Defaults to adding near end
                    print(f"      Applied MOD_ADDED: Added child to block '{parent_in_sim.key}'.")
                else:
                    print(f"    SIM MERGE ERROR (MOD_ADDED): Parent is invalid type {type(parent_in_sim).__name__}.")
            else:
                print(f"    SIM MERGE FAIL (MOD_ADDED): Mod node is absent.")


        elif chg_obj.type == 'MOD_DELETED':
            # Target node in sim tree *should* exist if Mod deleted something that Vanilla kept.
            # If Vanilla also deleted it, target_node_in_sim might be None, which is fine (already deleted).
            if target_node_in_sim is None:
                 print(f"      Applied MOD_DELETED: Target node already absent from sim tree.")
                 # Optional: Add comment to parent about deletion
                 _add_comment(parent_in_sim if isinstance(parent_in_sim, PdsNode) else None, f"{sim_comment_text} (target already absent)")
            else:
                if isinstance(parent_in_sim, list):
                    try:
                        parent_in_sim.remove(target_node_in_sim)
                        print(f"      Applied MOD_DELETED: Removed root node.")
                    except ValueError: print(f"    SIM MERGE ERROR (MOD_DEL Root): Target instance not found in root list for deletion.")
                elif isinstance(parent_in_sim, PdsBlock):
                     # We need the child's key/index identifier IN THE SIM TREE's context to remove.
                     # Same logic as MOD_MODIFIED replacement: find its current ID.
                     sim_child_id = None
                     current_key_group_count = 0
                     target_raw_id = PdsDiffer()._get_raw_identifier_for_node(target_node_in_sim)
                     for idx, child in enumerate(parent_in_sim.children):
                         child_raw_id = PdsDiffer()._get_raw_identifier_for_node(child)
                         if child is target_node_in_sim:
                            sim_child_id = f"{target_raw_id}___{current_key_group_count}"
                            break
                         if child_raw_id == target_raw_id:
                             current_key_group_count += 1

                     if sim_child_id:
                         if parent_in_sim.remove_child(sim_child_id):
                             print(f"      Applied MOD_DELETED: Removed child '{sim_child_id}' from block '{parent_in_sim.key}'.")
                         else:
                            print(f"    SIM MERGE WARN (MOD_DEL Block): PdsBlock.remove_child failed for '{sim_child_id}' in '{parent_in_sim.key}'.")
                     else:
                         print(f"    SIM MERGE ERROR (MOD_DEL Block): Could not determine child identifier for deletion in block '{parent_in_sim.key}'.")
                else:
                    print(f"    SIM MERGE ERROR (MOD_DELETED): Parent is invalid type {type(parent_in_sim).__name__}.")


        # Converged changes and Vanilla changes: already in New tree, just comment
        elif chg_obj.type in ('CONVERGED_MODIFICATION', 'MOD_ADDED_CONVERGED', 'VANILLA_MODIFIED', 'VANILLA_ADDED'):
             # Node should be in sim tree. It's the copy of chg_obj.new_node. target_node_in_sim should be found.
             if target_node_in_sim:
                 _add_comment(target_node_in_sim, sim_comment_text)
                 print(f"      Applied {chg_obj.type}: Added comment to node.")
             else:
                 print(f"    SIM MERGE WARN ({chg_obj.type}): Node not found in sim tree to add comment.")

        elif chg_obj.type == 'MOD_DELETED_VANILLA_ALSO_DELETED':
             # Both deleted it. Node should not be in sim tree (target_node_in_sim is None). Action: None. Comment parent.
             _add_comment(parent_in_sim if isinstance(parent_in_sim, PdsNode) else None, f"{sim_comment_text} for child slot '{child_key_for_parent_method}'")
             print(f"      Applied MOD_DELETED_VANILLA_ALSO_DELETED: Commented parent.")

        # Conflicts: apply a resolution strategy (prefer Mod unless specified)
        elif chg_obj.type.startswith('CONFLICT'):
            print(f"    Applying conflict resolution for {chg_obj.type}...")
            resolved_node_copy = None
            action_taken = "none" # "replaced", "added", "deleted", "kept_vanilla", "commented_parent"

            conflict_sim_comment = sim_comment_text

            # --- Determine Resolved Node or Action ---
            if chg_obj.type == 'CONFLICT_MODIFIED': # Mod modified, Vanilla modified differently
                # Prefer Mod's version
                if chg_obj.mod_node: resolved_node_copy = chg_obj.mod_node.copy(); conflict_sim_comment += "_MOD_CHOSEN"
                action_taken = "replaced" if target_node_in_sim else "none" # Must exist to replace

            elif chg_obj.type == 'CONFLICT_ADDITION': # Mod added, Vanilla added differently at same slot
                # Prefer Mod's version
                if chg_obj.mod_node: resolved_node_copy = chg_obj.mod_node.copy(); conflict_sim_comment += "_MOD_CHOSEN"
                action_taken = "added"

            elif chg_obj.type == 'CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED':
                # Mod deleted, Vanilla modified. Vanilla's is already in tree. Keep Vanilla's version.
                # Action is to keep the existing node and comment it. target_node_in_sim should be the V modified node copy.
                if target_node_in_sim:
                     _add_comment(target_node_in_sim, conflict_sim_comment + "_VANILLA_MOD_KEPT")
                     action_taken = "kept_vanilla"
                else:
                     print(f"    SIM MERGE WARN (CONFLICT_DELETION_MDVM): Target node not found in sim tree. Cannot comment.")
                     action_taken = "none"


            elif chg_obj.type == 'CONFLICT_DELETION_VANILLA_DELETED_MOD_MODIFIED':
                # Vanilla deleted, Mod modified. Vanilla's version is absent. Apply Mod's change as an addition.
                if chg_obj.mod_node:
                    resolved_node_copy = chg_obj.mod_node.copy()
                    conflict_sim_comment += "_MOD_MOD_APPLIED_TO_DELETED_SLOT"
                    action_taken = "added"

            # --- Apply Resolved Node or Action ---
            if resolved_node_copy:
                _add_comment(resolved_node_copy, conflict_sim_comment) # Add comment to the node being applied

            if action_taken == "replaced":
                if isinstance(parent_in_sim, list) and target_node_in_sim in parent_in_sim:
                    try:
                        idx = parent_in_sim.index(target_node_in_sim)
                        resolved_node_copy.indent_level=0
                        parent_in_sim[idx] = resolved_node_copy
                        print(f"      Applied {chg_obj.type}: Replaced root node at index {idx}.")
                    except ValueError: print(f"    SIM MERGE ERROR ({chg_obj.type} Root Replace): Target instance not found.")
                elif isinstance(parent_in_sim, PdsBlock) and target_node_in_sim in parent_in_sim.children:
                    # Need sim child ID to replace
                    sim_child_id = None
                    current_key_group_count = 0
                    target_raw_id = PdsDiffer()._get_raw_identifier_for_node(target_node_in_sim)
                    for idx, child in enumerate(parent_in_sim.children):
                        child_raw_id = PdsDiffer()._get_raw_identifier_for_node(child)
                        if child is target_node_in_sim:
                           sim_child_id = f"{target_raw_id}___{current_key_group_count}"
                           break
                        if child_raw_id == target_raw_id:
                            current_key_group_count += 1
                    if sim_child_id:
                        if parent_in_sim.replace_child(sim_child_id, resolved_node_copy):
                           print(f"      Applied {chg_obj.type}: Replaced child '{sim_child_id}' in block '{parent_in_sim.key}'.")
                        else:
                           print(f"    SIM MERGE WARN ({chg_obj.type} Block Replace): PdsBlock.replace_child failed for '{sim_child_id}'.")
                    else:
                        print(f"    SIM MERGE ERROR ({chg_obj.type} Block Replace): Could not determine child identifier.")
                else:
                    print(f"    SIM MERGE ERROR ({chg_obj.type} Replace): Parent or target not found or invalid type.")

            elif action_taken == "added":
                if resolved_node_copy:
                     if isinstance(parent_in_sim, list):
                         resolved_node_copy.indent_level = 0
                         parent_in_sim.append(resolved_node_copy) # TODO: Smarter insertion
                         print(f"      Applied {chg_obj.type}: Added root node.")
                     elif isinstance(parent_in_sim, PdsBlock):
                         parent_in_sim.add_child_at_appropriate_location(resolved_node_copy) # Defaults to adding near end
                         print(f"      Applied {chg_obj.type}: Added child to block '{parent_in_sim.key}'.")
                     else:
                        print(f"    SIM MERGE ERROR ({chg_obj.type} Added): Parent is invalid type {type(parent_in_sim).__name__}.")
                else:
                    print(f"    SIM MERGE FAIL ({chg_obj.type} Added): Resolved node copy is absent.")

            elif action_taken == "deleted": # Not currently used by conflict resolution, but for completeness
                 # Need sim child ID to remove
                 if target_node_in_sim:
                     if isinstance(parent_in_sim, list):
                         try: parent_in_sim.remove(target_node_in_sim)
                         except ValueError: print(f"    SIM MERGE ERROR ({chg_obj.type} Root Del): Target instance not found for deletion.")
                     elif isinstance(parent_in_sim, PdsBlock):
                         sim_child_id = None
                         current_key_group_count = 0
                         target_raw_id = PdsDiffer()._get_raw_identifier_for_node(target_node_in_sim)
                         for idx, child in enumerate(parent_in_sim.children):
                             child_raw_id = PdsDiffer()._get_raw_identifier_for_node(child)
                             if child is target_node_in_sim:
                                sim_child_id = f"{target_raw_id}___{current_key_group_count}"
                                break
                             if child_raw_id == target_raw_id:
                                 current_key_group_count += 1
                         if sim_child_id:
                             if parent_in_sim.remove_child(sim_child_id):
                                 print(f"      Applied {chg_obj.type}: Removed child '{sim_child_id}'.")
                             else:
                                print(f"    SIM MERGE WARN ({chg_obj.type} Block Del): PdsBlock.remove_child failed for '{sim_child_id}'.")
                         else:
                            print(f"    SIM MERGE ERROR ({chg_obj.type} Block Del): Could not determine child identifier.")
                     else:
                        print(f"    SIM MERGE ERROR ({chg_obj.type} Deleted): Parent is invalid type {type(parent_in_sim).__name__}.")
                 else:
                     print(f"    SIM MERGE FAIL ({chg_obj.type} Deleted): Target node not found in sim tree.")
            # "kept_vanilla" and "none" handled by commenting or requiring no action


        # Vanilla changes: already in the tree, just comment.
        elif chg_obj.type in ('VANILLA_MODIFIED', 'VANILLA_ADDED', 'VANILLA_DELETED'):
             # These changes originate from New Vanilla. They should already be present in the sim tree (unless deleted).
             # The target_node_in_sim should be found if it exists in New Vanilla.
             if chg_obj.type in ('VANILLA_MODIFIED', 'VANILLA_ADDED'):
                 if target_node_in_sim:
                     _add_comment(target_node_in_sim, sim_comment_text)
                     print(f"      Applied {chg_obj.type}: Added comment to node.")
                 else:
                     # This warning indicates target_node_in_sim wasn't found even though new_node was not None (for VM)
                     # or indicates the path-based parent lookup failed for VA (where new_node is the added node)
                     # If target_node_in_sim wasn't found but parent_in_sim was, maybe the node wasn't added correctly initially?
                     print(f"    SIM MERGE WARN ({chg_obj.type}): Target node not found in sim tree to add comment.")
             elif chg_obj.type == 'VANILLA_DELETED':
                 # target_node_in_sim should be None here as Vanilla deleted it. Just comment parent.
                 _add_comment(parent_in_sim if isinstance(parent_in_sim, PdsNode) else None, f"{sim_comment_text} for child slot '{child_key_for_parent_method}' (already absent)")
                 print(f"      Applied VANILLA_DELETED: Commented parent.")


    # End of _simulate_apply_change

    for change_item in changes:
        _simulate_apply_change(simulated_merged_nodes_root_list, change_item)

    simulated_merged_content = PdsParser._nodes_to_string(simulated_merged_nodes_root_list)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_SIMULATED_MERGED.txt")), simulated_merged_content)
    print(f"\n  Simulated merged content saved.")

    # --- Diff merged vs new vanilla ---
    print(f"\n--- DIFF: SIMULATED MERGED (Parser Output) vs NORMALIZED NEW VANILLA RAW for '{test_name}' ---")
    diff_filename = os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_MERGED_VS_NEW_RAW.diff"))
    with open(diff_filename, 'w', encoding='utf-8') as f_diff:
        diff_lines = list(difflib.unified_diff(
            normalized_new_raw.splitlines(keepends=True),
            simulated_merged_content.strip().splitlines(keepends=True),
            fromfile='NORMALIZED_NEW_VANILLA_RAW', tofile='SIMULATED_MERGED_PARSER_OUTPUT', lineterm=''
        ))
        if diff_lines:
            f_diff.writelines(diff_lines)
            print(f"  Diff saved to: {os.path.basename(diff_filename)}")
        else:
            # This message might be misleading if non-conflicting changes were expected but not applied.
            print("  SIMULATED MERGED (parser output) is identical to NORMALIZED NEW VANILLA (after accounting for applied changes).")
    print("-" * 80)

# --- Run Tests ---
print("Running diff tests with real CK3 files.")
run_and_print_diff("Siege Events (real files)",
                   SIEGE_EVENTS_OLD_VANILLA_PATH, SIEGE_EVENTS_MOD_PATH, SIEGE_EVENTS_NEW_VANILLA_PATH)
run_and_print_diff("00_tribal_innovations (real files)",
                   INNOVATIONS_OLD_VANILLA_PATH, INNOVATIONS_MOD_PATH, INNOVATIONS_NEW_VANILLA_PATH)
print("\n--- All Tests Complete ---")

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--------------------------------------------------------------------------------
Running diff tests with real CK3 files.

==================== RUNNING DIFF TEST: Siege Events (real files) ====================
Output files for this test will be saved to: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\test_output\Siege_Events_real_files
  Raw files saved.
  Reconstructed files saved.




--- DETECTED CHANGES for 'Siege Events (real files)' (27 changes) ---
PdsChange(Type='VANILLA_DELETED                              ', Path='siege.0002 =___23.__BLANK_LINE_____3', 
          ParentCtx='siege.0002 =___23', 
          Nodes=[O:PdsBlankLine L513 I4 Blank Line,
                 M:PdsBlankLine L513 I4 Blank Line,
                 N:ABSENT])
PdsChange(Type='VANILLA_DELETED                              ', Path='siege.0002 =___23.after =___4', 
          ParentCtx='siege.0002 =___2